# 使用 NVIDIA TensorRT-LLM 优化 OpenAI GPT-OSS 模型

本 notebook 将一步步介绍如何使用 NVIDIA 的 TensorRT-LLM 优化 `gpt-oss` 模型，以实现高性能推理。TensorRT-LLM 为用户提供了易用的 Python API 来定义大语言模型（LLM），并支持最先进的优化，可在 NVIDIA GPU 上高效执行推理。TensorRT-LLM 还包含用于创建 Python 和 C++ 运行时的组件，以高性能的方式编排推理执行。


TensorRT-LLM 支持以下两个模型：
- `gpt-oss-20b`
- `gpt-oss-120b`

在本指南中，我们将运行 `gpt-oss-20b`。如果你想尝试更大的模型或需要更多自定义，请参阅[这份](https://github.com/NVIDIA/TensorRT-LLM/blob/main/docs/source/blogs/tech_blog/blog9_Deploying_GPT_OSS_on_TRTLLM.md)部署指南。

注意：为了让模型正常工作，你的输入提示词应使用 [harmony 响应](http://cookbook.openai.com/articles/openai-harmony)格式，不过本指南并不要求这样做。

#### 在 NVIDIA Brev 上启动
你可以使用 [NVIDIA Brev](https://developer.nvidia.com/brev) 简化环境设置。点击下方按钮，即可在已预先配置好所需依赖的 Brev 实例上启动本项目。

部署完成后，点击“Open Notebook”按钮开始本指南

[![在 Brev 上启动](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-30i1YjHsRWT109HL6eYxLUeHIwF)

## 前提条件

### 硬件
要运行 gpt-oss-20b 模型，你需要一块至少有 20 GB 显存的 NVIDIA GPU。

推荐的 GPU：NVIDIA Hopper（例如 H100、H200）、NVIDIA Blackwell（例如 B100、B200）、NVIDIA RTX PRO、NVIDIA RTX 50 系列（例如 RTX 5090）。

### 软件
- CUDA Toolkit 12.8 或更高版本
- Python 3.12 或更高版本

## 安装 TensorRT-LLM

安装 TensorRT-LLM 有多种方式。在本指南中，我们将介绍如何使用 NVIDIA NGC 提供的预构建 Docker 容器，以及如何从源码构建。

如果你使用的是 NVIDIA Brev，可以跳过本节。

## 使用 NVIDIA NGC

从 [NVIDIA NGC](https://www.nvidia.com/en-us/gpu-cloud/) 拉取适用于 GPT-OSS 的预构建 [TensorRT-LLM 容器](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tensorrt-llm/containers/release/tags)。
这是最简单的入门方式，并能确保包含所有依赖。

```bash
docker pull nvcr.io/nvidia/tensorrt-llm/release:gpt-oss-dev
docker run --gpus all -it --rm -v $(pwd):/workspace nvcr.io/nvidia/tensorrt-llm/release:gpt-oss-dev
```

## 使用 Docker（从源码构建）

你也可以从源码构建 TensorRT-LLM 容器。
如果你想修改源码或使用自定义分支，这种方式会很有用。
详细说明请参阅[官方文档](https://github.com/NVIDIA/TensorRT-LLM/tree/feat/gpt-oss/docker)。

TensorRT-LLM 很快将可以通过 pip 安装

> 关于 GPU 架构的说明：首次运行模型时，TensorRT-LLM 会为你的特定 GPU 架构（例如 Hopper、Ada 或 Blackwell）构建一个优化后的引擎。如果你看到关于 GPU 的 CUDA 计算能力（例如 sm_90、sm_120）与已安装的 PyTorch 不兼容的警告，请确保你安装了最新的 NVIDIA 驱动程序，以及与你的 PyTorch 版本相匹配的 CUDA Toolkit 版本。

# 验证 TensorRT-LLM 安装

In [ ]:
from tensorrt_llm import LLM, SamplingParams

# 使用 TensorRT-LLM Python API

在下一个代码单元中，我们将演示如何使用 TensorRT-LLM Python API：
1. 从 Hugging Face 下载指定的模型权重（使用你的 HF_TOKEN 进行身份验证）。
2. 如果 TensorRT 引擎尚不存在，则为你的 GPU 架构自动构建它。
3. 加载模型并为推理做好准备。
4. 运行一个简单的文本生成示例，验证一切正常。

**注意**：首次运行可能需要几分钟，因为需要下载模型并构建引擎。
后续运行会快得多，因为引擎会被缓存。

In [ ]:
llm = LLM(model="openai/gpt-oss-20b")

In [ ]:
prompts = ["Hello, my name is", "The capital of France is"]
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)
for output in llm.generate(prompts, sampling_params):
    print(f"Prompt: {output.prompt!r}, Generated text: {output.outputs[0].text!r}")

# 结论与后续步骤
恭喜！你已经成功使用 TensorRT-LLM Python API 优化并运行了一个大语言模型。

在本 notebook 中，你学会了如何：
- 使用必要的依赖设置环境。
- 使用 `tensorrt_llm.LLM` API 从 Hugging Face Hub 下载模型。
- 自动构建针对你的 GPU 定制的高性能 TensorRT 引擎。
- 使用优化后的模型运行推理。


你可以探索更多高级功能，进一步提升性能和效率：

- 基准测试：尝试运行[基准测试](https://nvidia.github.io/TensorRT-LLM/performance/performance-tuning-guide/benchmarking-default-performance.html#benchmarking-with-trtllm-bench)，比较 TensorRT-LLM 引擎与原始 Hugging Face 模型的延迟和吞吐量。你可以遍历更多提示词并测量执行时间来完成比较。

- 量化：TensorRT-LLM [支持](https://github.com/NVIDIA/TensorRT-Model-Optimizer)多种量化技术（如 INT8 或 FP8），可在对准确率影响极小的情况下减小模型体积并加速推理。这对于在资源受限的硬件上部署模型是一项强大的功能。

- 使用 NVIDIA Dynamo 部署：在生产环境中，你可以使用 [NVIDIA Dynamo](https://docs.nvidia.com/dynamo/latest/) 部署 TensorRT-LLM 引擎，实现稳健、可扩展的多模型服务。

